# 3.3 — Agent Memory

Without memory, every agent turn is stateless — the agent forgets everything.

There are two types of memory:

| Type | What it stores | Lasts |
|------|---------------|-------|
| **Short-term** | The current conversation history | This session only |
| **Long-term** | Facts saved to a vector store | Across sessions |

```
Short-term:  [msg1, msg2, msg3, ...]  — grows with the conversation
Long-term:   Vector Store  ←→  Agent  — agent can save and recall facts
```

In [ ]:
!pip install langchain langchain-ollama langchain-community chromadb --quiet

## Part 1 — No Memory (the Problem)

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOllama(model='llama3.1', temperature=0)

# Stateless — each call has no history
def ask_stateless(question: str) -> str:
    response = llm.invoke([HumanMessage(content=question)])
    return response.content

print('Turn 1:', ask_stateless('My name is Alice and I love Python.'))
print()
print('Turn 2:', ask_stateless('What is my name?'))
print()
print('Notice: the model has no idea who you are in Turn 2!')

## Part 2 — Short-Term Memory (Conversation Buffer)

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage

@tool
def get_weather(city: str) -> str:
    """Returns current weather for a city."""
    data = {'london': 'Cloudy, 14°C', 'paris': 'Sunny, 22°C', 'tokyo': 'Humid, 28°C'}
    return data.get(city.lower(), f'No weather data for {city}')

tools = [get_weather]
tool_map = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

# Conversation history is stored in `memory` list
memory = [SystemMessage(content='You are a helpful assistant. Remember what the user tells you.')]

def chat_with_memory(user_input: str) -> str:
    """Stateful agent — appends every turn to memory."""
    memory.append(HumanMessage(content=user_input))

    while True:
        response = llm_with_tools.invoke(memory)
        memory.append(response)

        if not response.tool_calls:
            return response.content

        for tc in response.tool_calls:
            result = tool_map[tc['name']].invoke(tc['args'])
            memory.append(ToolMessage(content=str(result), tool_call_id=tc['id']))

# Multi-turn conversation
turns = [
    'Hi! My name is Alice and I live in London.',
    "What's the weather like where I live?",
    'What is my name?',
    'What city am I from?',
]

for turn in turns:
    print(f'User : {turn}')
    reply = chat_with_memory(turn)
    print(f'Agent: {reply}')
    print()

print(f'Memory has {len(memory)} messages total.')

## Part 3 — Sliding Window Memory

Long conversations overflow the context window.
A **sliding window** keeps only the last N messages.

In [ ]:
WINDOW_SIZE = 6  # keep last 6 messages (3 turns)

def chat_with_window(user_input: str, history: list, system_prompt: str) -> tuple:
    """Windowed memory — trims history to last WINDOW_SIZE messages."""
    history.append(HumanMessage(content=user_input))

    # Keep system prompt + last WINDOW_SIZE messages
    windowed = [SystemMessage(content=system_prompt)] + history[-WINDOW_SIZE:]

    response = llm.invoke(windowed)
    history.append(AIMessage(content=response.content))
    return response.content, history

history = []
system = 'You are a helpful assistant.'

conversation = [
    'My favourite colour is blue.',
    'I work as a data scientist.',
    'I enjoy hiking on weekends.',
    'I have a dog named Max.',
    'What is my favourite colour?',  # should still remember (in window)
]

for msg in conversation:
    reply, history = chat_with_window(msg, history, system)
    print(f'User : {msg}')
    print(f'Agent: {reply}')
    print(f'       [history length: {len(history)} messages]')
    print()

## Part 4 — Long-Term Memory with Vector Store

Long-term memory stores facts in a vector store.
The agent can **save** facts and **recall** them later — even after the session ends.

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document

embeddings = OllamaEmbeddings(model='llama3.1')
long_term_memory = Chroma(embedding_function=embeddings, collection_name='agent_memory')

def save_to_memory(fact: str):
    """Save a fact to long-term vector memory."""
    long_term_memory.add_documents([Document(page_content=fact)])
    print(f'  [saved to memory]: {fact}')

def recall_from_memory(query: str, k: int = 2) -> str:
    """Retrieve relevant facts from long-term memory."""
    docs = long_term_memory.similarity_search(query, k=k)
    if not docs:
        return 'No relevant memories found.'
    return '\n'.join(f'- {d.page_content}' for d in docs)

# Save some facts about the user
facts = [
    'The user is named Alice.',
    'Alice is a data scientist.',
    'Alice lives in London.',
    'Alice has a dog named Max.',
    'Alice favourite programming language is Python.',
]
for fact in facts:
    save_to_memory(fact)

print()
print('Recalling: "what is the user\'s job?"')
print(recall_from_memory('what is the user job profession'))
print()
print('Recalling: "does the user have a pet?"')
print(recall_from_memory('does the user have a pet or animal'))

In [ ]:
# Agent that uses long-term memory before answering

def agent_with_long_term_memory(question: str) -> str:
    # Step 1: retrieve relevant facts
    recalled = recall_from_memory(question)
    print(f'  [recalled]: {recalled}')

    # Step 2: inject into prompt
    messages = [
        SystemMessage(content=f"""You are a personalized assistant.
Use the following facts about the user to answer their question:

{recalled}

If the facts don't contain the answer, say you don't know."""),
        HumanMessage(content=question)
    ]
    response = llm.invoke(messages)
    return response.content

questions = [
    'What city do I live in?',
    'What is my dog\'s name?',
    'What programming language do I prefer?',
]

for q in questions:
    print(f'Q: {q}')
    print(f'A: {agent_with_long_term_memory(q)}')
    print()

## Summary

| Memory Type | Storage | Scope | Use when |
|-------------|---------|-------|----------|
| **None** | — | Single call | Stateless tasks |
| **Buffer** | Message list | Session | Short conversations |
| **Window** | Last N messages | Session | Long conversations |
| **Vector Store** | Chroma / disk | Persistent | User profiles, facts across sessions |